> **The stored results were removed, and this notebook is not the current source for this benchmark.**
>
> Its original recorded numbers were produced under polars 0.54.4 and compared
> `filter` (using `append_option`) against `builder` (using a manual `match`) —
> two variables changed at once, so the gap could not be attributed to either.
> Re-run as a 2x2 it turned out the streaming-filter abstraction is free and the
> whole cost belongs to `append_option`.
>
> The corrected experiment lives in [`../scripts/builder-vs-collect.rs`](../scripts/builder-vs-collect.rs),
> with results and provenance in [`../docs/builder-vs-collect-benchmark.md`](../docs/builder-vs-collect-benchmark.md).
> Keep this notebook for exploration; record conclusions there.

# EMA kernel micro-benchmark

Compares three implementations of the same EMA recurrence:

- **`builder`** — the current `calc_ema`: inline loop into a `PrimitiveChunkedBuilder`.
- **`collect`** — map over `ca.iter()` into `.collect()` (a different output-construction path).
- **`filter`** — per-element logic in a `#[inline]` struct method (`Ema::next(Option<f64>) -> Option<f64>`), fed into the **same builder** as `builder`. So `filter` vs `builder` isolates the cost of the streaming-filter abstraction; if they match, it's zero-cost.

The input has a **leading null**, so the nullable code path (validity bitmap) is exercised.

Run top to bottom. Dependencies come from `evcxr.toml` in this folder and compile on first run (slow once, then cached if `:cache` is set — see README.md). Optimization level comes from `evcxr.toml` too, set to cargo's release level so these timings are comparable with `cargo run --release`. Note: evcxr's `Instant`-based timing is noisy run-to-run — compare `min`, and treat differences under ~10% as noise.

In [ ]:
:dep polars
:dep itertools
:dep serde


In [ ]:
use polars::prelude::*;
use std::time::Instant;

In [ ]:
// Build N f64 values (reproducible random walk) with a leading null.
let n = 100_000usize;
let mut data: Vec<Option<f64>> = Vec::with_capacity(n);
data.push(None); // leading null -> forces the nullable path (validity bitmap)
let mut acc = 100.0f64;
let mut state: u64 = 0x9E3779B97F4A7C15;
for _ in 1..n {
    // xorshift64 for reproducible pseudo-random steps
    state ^= state << 13;
    state ^= state >> 7;
    state ^= state << 17;
    let u = (state >> 11) as f64 / ((1u64 << 53) as f64);
    acc += u - 0.5;
    data.push(Some(acc));
}
let ca: Float64Chunked = data.iter().copied().collect();
(ca.len(), ca.null_count())

In [ ]:
// Current approach: PrimitiveChunkedBuilder, append per element.
fn ema_builder(ca: &Float64Chunked, period: i64) -> Float64Chunked {
    let alpha = 2.0 / (period as f64 + 1.0);
    let mut ema = f64::NAN;
    let mut count: i64 = 0;
    let mut builder = PrimitiveChunkedBuilder::<Float64Type>::new("ema".into(), ca.len());
    for opt_val in ca.iter() {
        let Some(val) = opt_val else {
            ema = f64::NAN;
            count = 0;
            builder.append_null();
            continue;
        };
        if count == 0 { ema = val; } else { ema += alpha * (val - ema); }
        count += 1;
        if count >= period { builder.append_value(ema); } else { builder.append_null(); }
    }
    builder.finish()
}

In [ ]:
// Alternative: map into .collect() (TrustedLen), mirroring polars ewm_mean.
fn ema_collect(ca: &Float64Chunked, period: i64) -> Float64Chunked {
    let alpha = 2.0 / (period as f64 + 1.0);
    let mut ema = f64::NAN;
    let mut count: i64 = 0;
    ca.iter()
        .map(|opt_val| match opt_val {
            None => {
                ema = f64::NAN;
                count = 0;
                None
            }
            Some(val) => {
                if count == 0 { ema = val; } else { ema += alpha * (val - ema); }
                count += 1;
                (count >= period).then_some(ema)
            }
        })
        .collect()
}

In [ ]:
// Streaming-filter variant: per-element logic in a #[inline] struct method,
// fed into the SAME builder as `ema_builder` via append_option. So this differs
// from `ema_builder` only by the abstraction — if it matches, it's zero-cost.
struct Ema {
    period: i64,
    alpha: f64,
    ema: f64,
    count: i64,
}

impl Ema {
    #[inline]
    fn new(period: i64) -> Self {
        Self { period, alpha: 2.0 / (period as f64 + 1.0), ema: f64::NAN, count: 0 }
    }

    #[inline]
    fn next(&mut self, input: Option<f64>) -> Option<f64> {
        let Some(val) = input else {
            self.ema = f64::NAN;
            self.count = 0;
            return None;
        };
        if self.count == 0 { self.ema = val; } else { self.ema += self.alpha * (val - self.ema); }
        self.count += 1;
        (self.count >= self.period).then_some(self.ema)
    }
}

fn ema_filter(ca: &Float64Chunked, period: i64) -> Float64Chunked {
    let mut f = Ema::new(period);
    let mut builder = PrimitiveChunkedBuilder::<Float64Type>::new("ema".into(), ca.len());
    // append_option (ChunkedBuilder trait, via the prelude) = the value/null match.
    for x in ca.iter() {
        builder.append_option(f.next(x));
    }
    builder.finish()
}

In [ ]:
// All three must produce identical output. (A fn, not a closure: evcxr can't
// persist top-level closures.)
fn same(x: &Float64Chunked, y: &Float64Chunked) -> bool {
    x.len() == y.len()
        && x.iter().zip(y.iter()).all(|(p, q)| match (p, q) {
            (Some(p), Some(q)) => p == q,
            (None, None) => true,
            _ => false,
        })
}
let a: Float64Chunked = ema_builder(&ca, 20);
let b: Float64Chunked = ema_collect(&ca, 20);
let c: Float64Chunked = ema_filter(&ca, 20);
println!(
    "identical = {}  (len = {}, nulls = {})",
    same(&a, &b) && same(&a, &c), a.len(), a.null_count()
);

In [ ]:
fn bench<F: FnMut() -> Float64Chunked>(label: &str, runs: usize, mut f: F) {
    std::hint::black_box(f()); // warmup
    let mut best = f64::INFINITY;
    let mut total = 0.0f64;
    for _ in 0..runs {
        let t = Instant::now();
        let r = f();
        std::hint::black_box(&r);
        let us = t.elapsed().as_secs_f64() * 1e6;
        total += us;
        if us < best { best = us; }
    }
    println!("{:<9} min = {:8.1} us   mean = {:8.1} us   ({} runs)",
             label, best, total / runs as f64, runs);
}

In [ ]:
let period = 20i64;
bench("builder", 200, || ema_builder(&ca, period));
bench("collect", 200, || ema_collect(&ca, period));
bench("filter",  200, || ema_filter(&ca, period));